# 09 Permissions
Carry each page's visibility onto every chunk, filter by the asking user's role, and test for leaks.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-09"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: an access list on every chunk

In [2]:
import json, copy
import pandas as pd
from kit.rag import ACCESS, VectorIndex, chunk_pages, get_embedder, parse_corpus

pages = parse_corpus(CORPUS)
chunks = chunk_pages(pages, 120, 30)  # each chunk copies its page's access list
embedder = get_embedder()
index = VectorIndex(chunks, embedder)
print("Who may read each visibility:", ACCESS)
pd.DataFrame(chunks)[["id", "access"]].drop_duplicates("id").head(8)

Who may read each visibility: {'public': ['public', 'team', 'admin'], 'team': ['team', 'admin'], 'admin': ['admin']}


,id,access
0,qn-01#0,"[public, team, admin]"
1,qn-01#1,"[public, team, admin]"
2,qn-02#0,"[public, team, admin]"
3,qn-02#1,"[public, team, admin]"
4,qn-03#0,"[public, team, admin]"
5,qn-03#1,"[public, team, admin]"
6,qn-04#0,"[public, team, admin]"
7,qn-04#1,"[public, team, admin]"


Step 3: filter at query time by the asking user's role

In [3]:
query = "How do I refund a customer who was charged twice?"
for role in [None, "public", "admin"]:
    print(f"{str(role):7}", [c["id"] for c, _ in index.search(query, k=3, role=role)])

None    ['qn-13#0', 'qn-13#1', 'qn-10#0']
public  ['qn-06#0', 'qn-07#1', 'qn-05#1']
admin   ['qn-13#0', 'qn-13#1', 'qn-10#0']


Step 4: the two-user leak test: a public user must never get admin pages

In [4]:
ADMIN_PAGES = {p["id"] for p in pages if p["visibility"] == "admin"}
probes = [q["question"] for q in map(json.loads, open(CORPUS / "questions.jsonl"))]
probes += ["refund a double charge", "read the audit log", "delete a user's data", "admin console billing payments"]

def leaks(role, k=5):
    return [(q, c["id"]) for q in probes for c, _ in index.search(q, k=k, role=role) if c["page_id"] in ADMIN_PAGES]

print("Public user, filtered:", len(leaks("public")), "admin chunks retrieved")
print("Admin user, filtered:", len(leaks("admin")), "admin chunks retrieved (allowed)")
print("No filter at all:", len(leaks(None)), "admin chunks would reach a public user")
assert not leaks("public"), "LEAK: a public user retrieved an admin page"

# The same two users asking for an answer: the admin gets one with cited chunk ids, the public user does not.
from kit.rag import answer_with_citations
for user, role in [("ada", "admin"), ("pat", "public")]:
    a = answer_with_citations("How do I request a refund for being charged twice?", index.search, role)
    print(f"{user} ({role}): cites {a['cited_chunks'] or 'nothing'} | {a['text'][:80]}")
print("The same test runs in pytest: tests/test_permissions.py")

Public user, filtered: 0 admin chunks retrieved
Admin user, filtered: 22 admin chunks retrieved (allowed)
No filter at all: 22 admin chunks would reach a public user
ada (admin): cites ['qn-13#0'] | If a customer was charged twice for the same Quillnote purchase, you can issue a
pat (public): cites nothing | The sources do not answer this.
The same test runs in pytest: tests/test_permissions.py


Step 5: one permission change, and a stale copy in the index

In [5]:
CHANGED = "qn-12"          # Team shared templates: becomes admin only at minute 0
SYNC_EVERY_MIN = 15        # the job that copies access lists into the index
LAST_SYNC_MIN = -9         # it last ran 9 minutes before the change
CACHE_TTL_MIN = 5          # search results are cached for this long
truth = {p["id"]: ACCESS[p["visibility"]] for p in pages}

def index_at(minute):
    # The index holds the access lists from its last sync, not the live ones.
    last_sync = LAST_SYNC_MIN + ((minute - LAST_SYNC_MIN) // SYNC_EVERY_MIN) * SYNC_EVERY_MIN
    stale = copy.deepcopy(chunks)
    for c in stale:
        c["access"] = ACCESS["admin"] if (c["page_id"] == CHANGED and last_sync >= 0) else truth[c["page_id"]]
    return VectorIndex(stale, embedder), last_sync

def simulate(check_live_acl, minutes=40, query="How do shared templates work for a team workspace?"):
    live = dict(truth, **{CHANGED: ACCESS["admin"]})
    cache, leaked = {}, []
    for minute in range(minutes):
        if minute not in cache.get("valid", ()):
            idx, _ = index_at(minute)
            cache = {"hits": idx.search(query, k=3, role="team"), "valid": range(minute, minute + CACHE_TTL_MIN)}
        hits = cache["hits"]
        if check_live_acl:  # check the live access list again before showing anything
            hits = [(c, s) for c, s in hits if "team" in live[c["page_id"]]]
        if any(c["page_id"] == CHANGED for c, _ in hits):
            leaked.append(minute)
    return leaked

# Time here is simulated minute by minute, so the run takes seconds.
stale = simulate(check_live_acl=False)
window = f" (minute {stale[0]} to minute {stale[-1]})" if stale else ""
print(f"Index only: a team user still sees {CHANGED} for {len(stale)} minutes after the change{window}")
fixed = simulate(check_live_acl=True)
print(f"Index plus a live access check: {len(fixed)} minutes")
_ = (OUTPUT / "stale_window.json").write_text(json.dumps({
    "changed_page": CHANGED, "sync_every_min": SYNC_EVERY_MIN, "cache_ttl_min": CACHE_TTL_MIN,
    "stale_window_min_index_only": len(stale), "stale_minutes": stale, "stale_window_min_with_live_check": len(fixed)},
    indent=1))

Index only: a team user still sees qn-12 for 10 minutes after the change (minute 0 to minute 9)
Index plus a live access check: 0 minutes
